# Phase 5 — NB3: Joint Evaluation (MAMS)

**Goal:** End-to-end joint evaluation on MAMS dataset with two Stage 2 approaches:
1. **No-Retrieval** (baseline)
2. **Retrieval + Aux Loss**

Stage 1 uses 8 native MAMS categories, mapped to 5 SemEval categories for Stage 2 (which was trained on mapped data).

**Input:**
- `duclm318/p5-nb1-stage1-mams-data` — Stage 1 ckpt + 8-cat processed data
- `duclm318/p5-nb2-stage2-mams` — Stage 2 no-ret + auxloss checkpoints
- `duclm318/p5-embed-v6-mams` — MAMS embedding ckpt + 5-cat processed data

**Output:** `outputs_p5_nb3_mams/` — evaluation logs

## 0. Setup

In [ ]:
!pip install -q transformers faiss-cpu lxml scikit-learn pyyaml iterative-stratification

In [ ]:
import os, sys, json, shutil, subprocess, re

!git clone https://github.com/lucminhduc3108/Retrieval-ABSA.git /kaggle/working/repo
os.chdir('/kaggle/working/repo')
sys.path.insert(0, '/kaggle/working/repo')
print('Working dir:', os.getcwd())

In [ ]:
# Clone MAMS data (needed for data prep)
!git clone https://github.com/siat-nlp/MAMS-for-ABSA.git data/mams
print('MAMS data cloned.')

In [ ]:
import torch, gc
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
gc.collect()
torch.cuda.empty_cache()

## 0b. Experiment Configuration

In [ ]:
STAGE1_CONFIG = 'configs/stage1_mams_cataware.yaml'
STAGE1_CKPT_NAME = 'stage1_mams_cataware_best.pt'

EXPERIMENTS = [
    {"name": "No-Retrieval", "ckpt": "stage2_mams_noret_best.pt",
     "config": "configs/stage2_mams_noret.yaml", "tag": "noret",
     "no_retrieval": True},
    {"name": "Aux Loss",     "ckpt": "stage2_mams_auxloss_best.pt",
     "config": "configs/stage2_mams_auxloss.yaml", "tag": "auxloss",
     "no_retrieval": False},
]

print(f'Stage 1: {STAGE1_CONFIG}')
print(f'Experiments: {len(EXPERIMENTS)}')
for exp in EXPERIMENTS:
    print(f'  - {exp["name"]}: {exp["ckpt"]}')

## 0c. Wire Data & Checkpoints

In [ ]:
def find_input(name):
    for p in [f'/kaggle/input/{name}',
              f'/kaggle/input/datasets/duclm318/{name}',
              f'/kaggle/input/datasets/lcminhc/{name}']:
        if os.path.exists(p):
            return p
    raise FileNotFoundError(f'Dataset {name} not found')

NB1 = find_input('p5-nb1-stage1-mams-data')
NB2 = find_input('p5-nb2-stage2-mams')
EMB = find_input('p5-embed-v6-mams')

print(f'NB1: {NB1}')
print(f'NB2: {NB2} -> {os.listdir(NB2)}')
print(f'EMB: {EMB}')

# Stage 1 checkpoint
os.makedirs('checkpoints/stage1_mams', exist_ok=True)
shutil.copy(f'{NB1}/{STAGE1_CKPT_NAME}', 'checkpoints/stage1_mams/best.pt')
print(f'\nStage 1 ckpt: {STAGE1_CKPT_NAME}')

# Embedding checkpoint
os.makedirs('checkpoints/embedding_mams', exist_ok=True)
shutil.copy(f'{EMB}/embedding_best.pt', 'checkpoints/embedding_mams/best.pt')
print('Embedding ckpt wired.')

# Processed data: 8-cat from NB1 (for Stage 1), 5-cat from EMB (for Stage 2 + gold)
os.makedirs('data/processed_mams', exist_ok=True)
shutil.copy(f'{NB1}/category_detection.jsonl', 'data/processed_mams/category_detection.jsonl')
print('Category detection (8-cat) from NB1.')
shutil.copy(f'{EMB}/processed_mams/sentiment_records.jsonl', 'data/processed_mams/sentiment_records.jsonl')
print('Sentiment records (5-cat mapped) from EMB.')
shutil.copy(f'{EMB}/processed_mams/classification.jsonl', 'data/processed_mams/classification.jsonl')
print('Classification records from EMB.')

# Stage 2 checkpoints
wired = []
for exp in EXPERIMENTS:
    src = f'{NB2}/{exp["ckpt"]}'
    dst_dir = f'checkpoints/stage2_mams_{exp["tag"]}'
    os.makedirs(dst_dir, exist_ok=True)
    if os.path.exists(src):
        shutil.copy(src, f'{dst_dir}/best.pt')
        wired.append(exp["name"])
        print(f'  {exp["ckpt"]}: {os.path.getsize(src)/1e6:.1f} MB')
    else:
        print(f'  WARNING: {exp["ckpt"]} NOT FOUND')
print(f'\nWired {len(wired)}/{len(EXPERIMENTS)} checkpoints: {wired}')

# Build FAISS index from sentiment records (5-cat, matching Stage 2 training)
os.makedirs('indexes/mams', exist_ok=True)
!python scripts/03_build_index.py \
    --embedding_ckpt checkpoints/embedding_mams/best.pt \
    --input data/processed_mams/sentiment_records.jsonl \
    --out_dir indexes/mams/

## 1. Run Evaluations

In [ ]:
os.makedirs('logs', exist_ok=True)
results = []

for exp in EXPERIMENTS:
    ckpt_path = f'checkpoints/stage2_mams_{exp["tag"]}/best.pt'
    if not os.path.exists(ckpt_path):
        print(f'\n=== SKIP {exp["name"]} — checkpoint missing ===')
        continue

    print(f'\n{"=" * 60}')
    print(f'Evaluating: {exp["name"]}')
    print(f'{"=" * 60}')

    gc.collect()
    torch.cuda.empty_cache()

    cmd = [
        'python', 'scripts/05_evaluate_joint.py',
        '--stage1_ckpt', 'checkpoints/stage1_mams/best.pt',
        '--stage2_ckpt', ckpt_path,
        '--embedding_ckpt', 'checkpoints/embedding_mams/best.pt',
        '--index_dir', 'indexes/mams/',
        '--stage1_config', STAGE1_CONFIG,
        '--stage2_config', exp['config'],
        '--retrieval_config', 'configs/retrieval_v2.yaml',
        '--pred_strategy', 'per_category',
        '--category_map', 'mams_to_semeval',
    ]
    if exp.get("no_retrieval"):
        cmd.append('--no_retrieval')

    result = subprocess.run(cmd, capture_output=True, text=True)
    print(result.stdout[-3000:] if len(result.stdout) > 3000 else result.stdout)
    if result.returncode != 0:
        print(f'STDERR: {result.stderr[-1000:]}')
        continue

    # Rename output log
    tag = "noret" if exp.get("no_retrieval") else "retrieval"
    src_log = f'logs/joint_eval_{tag}.md'
    dst_log = f'logs/joint_eval_mams_{exp["tag"]}.md'
    if os.path.exists(src_log):
        shutil.copy(src_log, dst_log)

    # Parse metrics from stdout
    metrics = {'name': exp['name']}
    for line in result.stdout.split('\n'):
        if 'Cat F1=' in line:
            m = re.search(r'Cat F1=([\d.]+).*Joint F1=([\d.]+).*Sent Acc\|CC=([\d.]+).*Sent MacF1\|CC=([\d.]+)', line)
            if m:
                metrics['cat_f1'] = float(m.group(1))
                metrics['joint_f1'] = float(m.group(2))
                metrics['sent_acc'] = float(m.group(3))
                metrics['sent_macf1'] = float(m.group(4))
    results.append(metrics)
    print(f'\n-> Joint F1={metrics.get("joint_f1", "?")}, Sent Acc={metrics.get("sent_acc", "?")}')

## 2. Comparison Table

In [ ]:
print('=' * 80)
print('MAMS EVALUATION — No-Retrieval vs Aux Loss')
print('=' * 80)
print(f'{"Approach":<18s} | {"Cat F1":>7s} | {"Joint F1":>8s} | {"Sent Acc|CC":>11s} | {"Sent MacF1":>10s}')
print('-' * 65)

noret_jf1 = None
for r in results:
    jf1 = r.get('joint_f1', 0)
    if r['name'] == 'No-Retrieval':
        noret_jf1 = jf1
    delta_str = ''
    if noret_jf1 is not None and r['name'] != 'No-Retrieval':
        delta = jf1 - noret_jf1
        delta_str = f'  ({delta:+.4f})'
    print(f'{r["name"]:<18s} | {r.get("cat_f1", 0):>7.4f} | {jf1:>8.4f}{delta_str:>10s} | '
          f'{r.get("sent_acc", 0):>11.4f} | {r.get("sent_macf1", 0):>10.4f}')
print('-' * 65)

# Compare with SemEval 2014 results
print('\n--- Cross-Dataset Comparison (SemEval 2014 reference) ---')
print(f'{"Dataset":<18s} | {"Cat F1":>7s} | {"No-Ret JF1":>10s} | {"Ret JF1":>8s} | {"Gap":>8s}')
print('-' * 60)
sem_noret = 0.7696
sem_ret = 0.7273
print(f'{"SemEval 2014":<18s} | {0.8564:>7.4f} | {sem_noret:>10.4f} | {sem_ret:>8.4f} | {sem_ret-sem_noret:>+8.4f}')
mams_noret = next((r.get('joint_f1', 0) for r in results if r['name'] == 'No-Retrieval'), 0)
mams_ret = next((r.get('joint_f1', 0) for r in results if r['name'] == 'Aux Loss'), 0)
mams_cat = next((r.get('cat_f1', 0) for r in results if r['name'] == 'No-Retrieval'), 0)
print(f'{"MAMS":<18s} | {mams_cat:>7.4f} | {mams_noret:>10.4f} | {mams_ret:>8.4f} | {mams_ret-mams_noret:>+8.4f}')

## 3. Individual Results

In [ ]:
for exp in EXPERIMENTS:
    log_path = f'logs/joint_eval_mams_{exp["tag"]}.md'
    if os.path.exists(log_path):
        print(f'\n{"=" * 60}')
        print(f'{exp["name"]}')
        print(f'{"=" * 60}')
        with open(log_path) as f:
            print(f.read())
    else:
        print(f'\n{exp["name"]}: no results')

## 4. Save Outputs

In [ ]:
output_dir = '/kaggle/working/outputs_p5_nb3_mams'
os.makedirs(output_dir, exist_ok=True)

if os.path.exists('logs'):
    shutil.copytree('logs', f'{output_dir}/logs', dirs_exist_ok=True)
    print('logs/ copied')

# Save results summary
with open(f'{output_dir}/results_summary.json', 'w') as f:
    json.dump(results, f, indent=2)
    print('results_summary.json saved')

print(f'\nOutputs saved to {output_dir}')